# Example One

## Instrument information

https://notes.cibc.com/#/productdetail/9247

<img src="./imgs/autocallable_7.jpg" alt="drawing" width="800"/>

In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd
import numpy as np

from inception.instruments.constant_parameters import NOTES_PARAMETERS
from inception.utils import hide_code
from inception.instruments import ReferenceIndexReturnSimulator, AutoCallableNote
from inception.instruments import ReferenceMultiIndexReturnSimulator, MultiAssetAutoCallableNote

hide_code()

**Assumption: The underlying index issued price is 775; Number of simulation path is 100,000; Principal amount is 100.**

## 1. Value of Autocallable Notes

In [2]:
hide_code()

dates = ['2023-09-18', '2024-03-08', '2028-05-01']
issued_price = 775

params = [(775, 0.3, 0.05), (790, 0.2, 0.03), (750, 0.4, 0.06)]

path_num = 100_000
d_s = 0.02
parameters = NOTES_PARAMETERS['CIBC']
values = []
deltas = []
gammas = []

for i, date in enumerate(dates):
    values.append([])
    deltas.append([])
    gammas.append([])
    for p in params:
        simulator = ReferenceIndexReturnSimulator(issued_price, p[0], p[2], p[1], delta_s=d_s)
        notes = AutoCallableNote(date,
                                 simulator, 
                                 parameters['valuation dates'],
                                 parameters['coupon dates'],
                                 parameters['call dates'],
                                 coupon_barrier=parameters['coupon barrier'],
                                 call_barrier=parameters['call barrier'],
                                 principal_barrier=parameters['principal barrier'],
                                 coupon_amount=parameters['coupon amount'],
                                 free_rate=p[2],
                                 n_paths=path_num)
        values[i].append(notes.value)
        deltas[i].append(notes.delta.values[0])
        gammas[i].append(notes.gamma.values[0])

pd.DataFrame(values, index=dates,
             columns=[f'spot price={p[0]}, volatility={p[1]}, free rate={p[2]}' for p in params]).T.round(5)

,2023-09-18,2024-03-08,2028-05-01
"spot price=775, volatility=0.3, free rate=0.05",100.06562,100.66351,100.48940
"spot price=790, volatility=0.2, free rate=0.03",107.90341,102.11560,106.17068
"spot price=750, volatility=0.4, free rate=0.06",92.50226,94.27403,94.14807


## 2. Delta of Autocallable Notes

In [3]:
hide_code()

pd.DataFrame(deltas, index=dates,
             columns=[f'spot price={p[0]}, volatility={p[1]}, free rate={p[2]}' for p in params]).T.round(5)

,2023-09-18,2024-03-08,2028-05-01
"spot price=775, volatility=0.3, free rate=0.05",0.02737,0.01630,0.02573
"spot price=790, volatility=0.2, free rate=0.03",-0.02706,-0.13244,-0.02100
"spot price=750, volatility=0.4, free rate=0.06",0.05195,0.10533,0.05022


## 3. Gamma of Autocallable Notes

In [4]:
hide_code()

pd.DataFrame(gammas, index=dates,
             columns=[f'spot price={p[0]}, volatility={p[1]}, free rate={p[2]}' for p in params]).T.round(5)

,2023-09-18,2024-03-08,2028-05-01
"spot price=775, volatility=0.3, free rate=0.05",-0.00016,-0.00083,-0.00031
"spot price=790, volatility=0.2, free rate=0.03",-0.00004,0.00828,-0.00007
"spot price=750, volatility=0.4, free rate=0.06",-0.00022,0.00124,-0.00038


.

.

# Example Two

## Instrument information

https://www.sec.gov/Archives/edgar/data/1045520/000110465924024467/tm244771d24_424b2.htm

<img src="./imgs/autocallable_4y.jpg" alt="drawing" width="1000"/>

**Assumption: The underlying indexes issued price: DIA = 382.82, SPY=494.08, XLV=142.86; Number of simulation path is 100,000; Principal amount is 1,000**

**The Correlation Matrix:**

In [5]:
hide_code()

corr_matrix = np.array(
        [[1.        , 0.96432854, 0.80799003],
        [0.96432854, 1.        , 0.8267037 ],
        [0.80799003, 0.8267037 , 1.        ]]
    )
pd.DataFrame(corr_matrix, index=['DIA', 'SPY', 'XLV'], columns=['DIA', 'SPY', 'XLV'])

,DIA,SPY,XLV
DIA,1.000000,0.964329,0.807990
SPY,0.964329,1.000000,0.826704
XLV,0.807990,0.826704,1.000000


## 1. Value of Autocallable Notes

In [6]:
hide_code()

dates = ['2024-02-14', '2024-08-13', '2026-07-01']
issued_price = [382.82, 494.08, 142.86]

params = [([382, 494, 142], [0.3, 0.3, 0.3], 0.05), 
          ([400, 500, 150], [0.2, 0.2, 0.2], 0.06), 
          ([350, 450, 120], [0.1, 0.1, 0.1], 0.02), ]

path_num = 100_000
d_s = 0.02
parameters = NOTES_PARAMETERS['CIBC Multi Assets One']
values = []
deltas = []
gammas = []

for i, date in enumerate(dates):
    values.append([])
    deltas.append([])
    gammas.append([])
    for p in params:
        simulator = ReferenceMultiIndexReturnSimulator(issued_price, p[0], p[2], p[1], corr_matrix, d_s)
        notes = MultiAssetAutoCallableNote(date,
                                           ['A', 'B', 'C'],
                                           simulator, 
                                           parameters['valuation dates'],
                                           parameters['coupon dates'],
                                           parameters['call dates'],
                                           coupon_barrier=parameters['coupon barrier'],
                                           call_barrier=parameters['call barrier'],
                                           principal_barrier=parameters['principal barrier'],
                                           coupon_amount=parameters['coupon amount'] * 10,
                                           notional=1000,
                                           free_rate=p[2],
                                           n_paths=path_num)
        values[i].append(notes.value)
        deltas[i].append(notes.delta.values)
        gammas[i].append(notes.gamma.values)

pd.DataFrame(values, index=dates,
             columns=[f'spot price={p[0]}, volatility={p[1]}, free rate={p[2]}' for p in params]).T.round(5)

,2024-02-14,2024-08-13,2026-07-01
"spot price=[382, 494, 142], volatility=[0.3, 0.3, 0.3], free rate=0.05",904.70182,960.75083,959.03500
"spot price=[400, 500, 150], volatility=[0.2, 0.2, 0.2], free rate=0.06",990.84759,1021.09165,1011.04559
"spot price=[350, 450, 120], volatility=[0.1, 0.1, 0.1], free rate=0.02",1131.80646,1145.48731,1091.00705


## 2. Delta of Autocallable Notes

In [7]:
hide_code()

pd.DataFrame(deltas, index=dates,
             columns=[f'spot price={p[0]}, volatility={p[1]}, free rate={p[2]}' for p in params]).T.applymap(
    lambda x_list:[round(x, 5) for x in x_list])

,2024-02-14,2024-08-13,2026-07-01
"spot price=[382, 494, 142], volatility=[0.3, 0.3, 0.3], free rate=0.05","[0.45973, 0.29451, 1.7329]","[1.57037, 1.02106, 6.458]","[0.47481, 0.30325, 1.83562]"
"spot price=[400, 500, 150], volatility=[0.2, 0.2, 0.2], free rate=0.06","[0.13085, 0.21695, 0.57973]","[0.01213, 0.45686, 0.06641]","[0.05486, 0.10964, 0.23328]"
"spot price=[350, 450, 120], volatility=[0.1, 0.1, 0.1], free rate=0.02","[0.14315, 0.11893, 3.26403]","[0.12347, 0.10166, 3.36282]","[0.05934, 0.05042, 3.23664]"


## 3. Gamma of Autocallable Notes

In [8]:
hide_code()

pd.DataFrame(gammas, index=dates,
             columns=[f'spot price={p[0]}, volatility={p[1]}, free rate={p[2]}' for p in params]).T.applymap(
    lambda x_list:[round(x, 5) for x in x_list])

,2024-02-14,2024-08-13,2026-07-01
"spot price=[382, 494, 142], volatility=[0.3, 0.3, 0.3], free rate=0.05","[-0.0155, -0.00939, -0.03448]","[-0.27384, -0.16263, -1.26813]","[-0.03114, -0.01589, -0.14263]"
"spot price=[400, 500, 150], volatility=[0.2, 0.2, 0.2], free rate=0.06","[-0.00709, -0.00506, -0.05237]","[-0.00041, -0.06236, -0.0031]","[-0.00399, -0.00534, -0.03204]"
"spot price=[350, 450, 120], volatility=[0.1, 0.1, 0.1], free rate=0.02","[-0.00964, -0.0084, -0.70092]","[-0.01031, -0.00642, -0.67649]","[-0.00777, -0.00488, -0.58122]"
